# ASA-Transformer vs Dense Transformer — GPU Benchmark (Kaggle T4 x2)

**Adaptive Selective Attention (ASA)** — Buenahora Ormaza (2026)  

Benchmark escalado: secuencias de **1024 tokens**, modelos **~500K parámetros** en GPU T4.  
Métricas: velocidad (tok/sec), perplexity, VRAM pico, consistencia entre runs.

In [ ]:
# ── Cell 1: Environment ──────────────────────────────────────────────
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pymbbo'])

try:
    import triton
    print(f'Triton: {triton.__version__}')
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'triton'])

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}  VRAM={p.total_memory/1e9:.1f} GB')

In [ ]:
# ── Cell 2: Triton + SDPA verification ──────────────────────────────
import math
import torch
import torch.nn.functional as F

TRITON_AVAILABLE = False
try:
    if torch.cuda.is_available():
        import triton, triton.language as tl
        TRITON_AVAILABLE = True
        print(f'[OK] Triton kernel backend ACTIVE  v{triton.__version__}')
except Exception as e:
    print(f'[WARN] Triton not available: {e}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Verify FlashAttention-2 via SDPA
if torch.cuda.is_available() and hasattr(F, 'scaled_dot_product_attention'):
    q_ = torch.randn(2,4,64,32, device='cuda')
    out_ = F.scaled_dot_product_attention(q_, q_, q_, is_causal=True)
    print(f'[OK] FlashAttention-2 (SDPA) available  shape={out_.shape}')
    del q_, out_
    torch.cuda.empty_cache()

print(f'\nDevice: {DEVICE}')

In [ ]:
# ── Cell 3: Memory-efficient ASA implementation (standalone) ─────────
# NOTE: This implementation corrects the O(N²) gather bug present in
# older builds. The flatten-index gather keeps memory at O(N·A·D)
# instead of the catastrophic O(N·N_total·D).

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional, Tuple, List


# ── Gather helper ────────────────────────────────────────────────────
def _efficient_gather_kv(
    k: torch.Tensor,        # (B, H, N_total, D)
    v: torch.Tensor,        # (B, H, N_total, D)
    sel: torch.Tensor,      # (B, N, A)  — selection indices
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Memory-efficient K/V gather using the flatten-index trick.

    Standard approach (WRONG — OOM):
        k.unsqueeze(2).expand(B, H, N, N_total, D)  → materializes N·N_total·D

    This approach (CORRECT):
        Flatten N_total·D, build a flat offset index, gather once.
        Memory cost: O(B·H·N·A·D)  instead of  O(B·H·N·N_total·D)
    """
    B, H, N_total, D = k.shape
    N, A = sel.shape[1], sel.shape[2]

    # (B, H, N_total, D) → (B*H, N_total*D)
    k_flat = k.reshape(B * H, N_total * D)
    v_flat = v.reshape(B * H, N_total * D)

    # sel: (B, N, A) → (B*H, N, A)  — expand head dim (view only, no copy)
    sel_bh = sel.unsqueeze(1).expand(B, H, N, A).reshape(B * H, N, A)

    # Build flat indices:  offset[bh, n, a, d] = sel[bh, n, a] * D + d
    d_off   = torch.arange(D, device=k.device).view(1, 1, 1, D)
    flat_idx = (sel_bh.unsqueeze(-1) * D + d_off).reshape(B * H, N * A * D)

    k_gathered = torch.gather(k_flat, 1, flat_idx).reshape(B, H, N, A, D)
    v_gathered = torch.gather(v_flat, 1, flat_idx).reshape(B, H, N, A, D)
    return k_gathered, v_gathered


# ── Pass-2 attention dispatcher ──────────────────────────────────────
def _pass2_attn(
    q:          torch.Tensor,   # (B, H, N, D)
    k_gathered: torch.Tensor,   # (B, H, N, A, D)
    v_gathered: torch.Tensor,   # (B, H, N, A, D)
    scale:      float,
) -> torch.Tensor:              # (B, H, N, D)
    """Renormalized sparse attention over selected tokens."""
    scores  = (q.unsqueeze(-2) * k_gathered).sum(-1) * scale  # (B,H,N,A)
    weights = F.softmax(scores, dim=-1)
    return (weights.unsqueeze(-1) * v_gathered).sum(-2)         # (B,H,N,D)


# ── Core ASA Module ──────────────────────────────────────────────────
class AdaptiveSelectiveAttention(nn.Module):
    """
    ASA: Score-gated Two-Pass Causal Attention with Selection Sharing.

    Pass 1  (router only): dense causal scoring → importance I_i(j).
    Selection:             top-a tokens per query + self-token.
    Pass 2  (all layers):  renormalized sparse attention over selection T.
    Memory:                O(N·A·D) for Pass-2 gather (not O(N·N_total·D)).
    """
    def __init__(self, d_model: int, nhead: int, max_a: int = 64,
                 is_router: bool = True, dropout: float = 0.0,
                 margin_delta: float = 0.1):
        super().__init__()
        assert d_model % nhead == 0
        self.d_model     = d_model
        self.nhead       = nhead
        self.head_dim    = d_model // nhead
        self.max_a       = max_a
        self.is_router   = is_router
        self.scale       = 1.0 / math.sqrt(self.head_dim)
        self.margin_delta = margin_delta
        self.q_proj  = nn.Linear(d_model, d_model)
        self.k_proj  = nn.Linear(d_model, d_model)
        self.v_proj  = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout  = nn.Dropout(dropout)

    def forward(
        self, x: torch.Tensor,
        selection_indices: Optional[torch.Tensor] = None,
        max_a: Optional[int] = None,
        return_aux_loss: bool = False,
        kv_cache: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
    ):
        B, N, _ = x.shape
        H, D    = self.nhead, self.head_dim

        q = self.q_proj(x).view(B, N, H, D).transpose(1, 2)  # (B,H,N,D)
        k = self.k_proj(x).view(B, N, H, D).transpose(1, 2)
        v = self.v_proj(x).view(B, N, H, D).transpose(1, 2)

        if kv_cache is not None:
            ck, cv = kv_cache
            k = torch.cat([ck, k], dim=2)
            v = torch.cat([cv, v], dim=2)
        new_kv = (k, v)

        N_total = k.shape[2]
        budget  = max_a if max_a is not None else self.max_a

        # ── Full attention fallback (FlashAttention-2 on CUDA) ──────
        if budget >= N_total:
            if hasattr(F, 'scaled_dot_product_attention') and x.is_cuda:
                dp  = self.dropout.p if self.training else 0.0
                out = F.scaled_dot_product_attention(q, k, v,
                          is_causal=(N == N_total), dropout_p=dp)
            else:
                s = (q @ k.transpose(-2, -1)) * self.scale
                if N == N_total:
                    m = torch.triu(torch.full((N, N), float('-inf'), device=x.device), 1)
                    s = s + m[None, None]
                out = F.softmax(s, dim=-1) @ v
            out = self.out_proj(out.transpose(1, 2).contiguous().view(B, N, self.d_model))
            fi  = torch.arange(N_total, device=x.device).view(1,1,-1).expand(B, N, -1)
            return out, fi, None, new_kv

        aux_loss = None

        # ── Router Layer: Pass-1 dense scoring ──────────────────────
        if self.is_router or selection_indices is None:
            raw = (q @ k.transpose(-2, -1)) * self.scale            # (B,H,N,N_total)
            cm  = torch.triu(torch.full((N, N_total), float('-inf'), device=x.device), 1)
            imp = F.softmax(raw + cm[None, None], dim=-1).mean(1)   # (B,N,N_total)

            masked_imp = imp.masked_fill(cm[None] == float('-inf'), -1e9)
            a_cap      = min(budget, N_total)
            _, top_idx = torch.topk(masked_imp, k=a_cap, dim=-1, sorted=False)

            # Always include the current token i
            self_idx  = torch.arange(N_total - N, N_total, device=x.device) \
                             .view(1, -1, 1).expand(B, N, 1)
            selection_indices = torch.cat([top_idx, self_idx], dim=-1)  # (B,N,a_cap+1)

            # Margin loss (optional)
            if return_aux_loss:
                sel_s   = torch.gather(imp, -1, selection_indices)
                min_sel = sel_s.min(-1).values
                valid   = cm[None].expand(B, N, N_total) != float('-inf')
                smask   = torch.zeros((B, N, N_total), dtype=torch.bool, device=x.device)
                smask.scatter_(-1, selection_indices, True)
                non_sel = imp.masked_fill(~(valid & ~smask), -1e9)
                mns     = non_sel.max(-1).values
                mns     = torch.where(mns == -1e9, torch.zeros_like(mns), mns)
                aux_loss = F.relu(self.margin_delta - min_sel + mns).mean()

        # ── Pass-2: Memory-efficient sparse attention ────────────────
        k_gath, v_gath = _efficient_gather_kv(k, v, selection_indices)  # (B,H,N,A,D)
        out = _pass2_attn(q, k_gath, v_gath, self.scale)                # (B,H,N,D)
        out = self.out_proj(out.transpose(1, 2).contiguous().view(B, N, self.d_model))
        return out, selection_indices, aux_loss, new_kv


# ── Building blocks ──────────────────────────────────────────────────
class FFN(nn.Module):
    def __init__(self, d, ff=None, dr=0.0):
        super().__init__()
        ff = ff or d * 4
        self.net = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(dr), nn.Linear(ff, d))
    def forward(self, x): return self.net(x)


class ASABlock(nn.Module):
    def __init__(self, d_model, nhead, max_a=64, is_router=True, dropout=0.0):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = AdaptiveSelectiveAttention(d_model, nhead, max_a, is_router, dropout)
        self.ln2  = nn.LayerNorm(d_model)
        self.ffn  = FFN(d_model, dropout=dropout)

    def forward(self, x, sel=None, max_a=None, ret_aux=False, kvc=None):
        a, sel, aux, kvc = self.attn(self.ln1(x), sel, max_a, ret_aux, kvc)
        x = x + a
        x = x + self.ffn(self.ln2(x))
        return x, sel, aux, kvc


class ASALayerGroup(nn.Module):
    """1 Router + (g-1) Followers sharing selection T."""
    def __init__(self, g, d_model, nhead, max_a=64, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            ASABlock(d_model, nhead, max_a, is_router=(i == 0), dropout=dropout)
            for i in range(g)
        ])

    def forward(self, x, max_a=None, ret_aux=False, g_kvc=None):
        shared_sel, aux_total, new_kvc = None, None, []
        for i, layer in enumerate(self.layers):
            kvc_i = g_kvc[i] if g_kvc else None
            x, sel, aux, kvc_new = layer(x, shared_sel, max_a, ret_aux, kvc_i)
            if i == 0: shared_sel = sel
            if aux is not None:
                aux_total = aux if aux_total is None else aux_total + aux
            if kvc_new is not None: new_kvc.append(kvc_new)
        return x, aux_total, new_kvc


class ASATransformerGPT(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=6,
                 max_seq_len=1024, group_size=2, max_a=64, dropout=0.0):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop    = nn.Dropout(dropout)
        groups, rem  = [], num_layers
        while rem > 0:
            g = min(group_size, rem)
            groups.append(ASALayerGroup(g, d_model, nhead, max_a, dropout))
            rem -= g
        self.groups   = nn.ModuleList(groups)
        self.final_ln = nn.LayerNorm(d_model)
        self.lm_head  = nn.Linear(d_model, vocab_size)

    def forward(self, x, max_a=None, return_aux_loss=False, kv_caches=None):
        B, N = x.shape
        past = kv_caches[0][0][0].shape[2] if kv_caches and kv_caches[0] else 0
        pos  = torch.arange(past, past + N, device=x.device).unsqueeze(0)
        h    = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        aux_total, new_kvc = None, []
        for i, grp in enumerate(self.groups):
            gc = kv_caches[i] if kv_caches else None
            h, aux, gkvc = grp(h, max_a, return_aux_loss, gc)
            if aux is not None:
                aux_total = aux if aux_total is None else aux_total + aux
            if gkvc: new_kvc.append(gkvc)
        logits = self.lm_head(self.final_ln(h))
        return (logits, aux_total) if return_aux_loss else logits

    @torch.no_grad()
    def generate(self, prompt, max_new_tokens=60, max_a=None):
        self.eval()
        dev  = prompt.device
        curr = prompt.clone()
        B, N = curr.shape
        pos  = torch.arange(N, device=dev).unsqueeze(0)
        h    = self.drop(self.tok_emb(curr) + self.pos_emb(pos))
        kvc  = []
        for grp in self.groups:
            h, _, gkvc = grp(h, max_a)
            kvc.append(gkvc)
        curr = torch.cat([curr, self.lm_head(self.final_ln(h))[:, -1:].argmax(-1, keepdim=True)], 1)

        for _ in range(max_new_tokens - 1):
            if curr.size(1) >= self.max_seq_len: break
            st  = curr[:, -1:]
            pl  = curr.size(1) - 1
            pos = torch.tensor([[pl]], device=dev)
            h   = self.drop(self.tok_emb(st) + self.pos_emb(pos))
            nkvc = []
            for i, grp in enumerate(self.groups):
                h, _, gkvc = grp(h, max_a, g_kvc=kvc[i])
                nkvc.append(gkvc)
            kvc  = nkvc
            curr = torch.cat([curr, self.lm_head(self.final_ln(h))[:, -1:].argmax(-1, keepdim=True)], 1)
        return curr


# ── Standard Dense GPT baseline ─────────────────────────────────────
class DenseGPT(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=6,
                 max_seq_len=1024, dropout=0.0):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        enc = nn.TransformerEncoderLayer(d_model, nhead, d_model*4, dropout,
                                          batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers)
        self.final_ln    = nn.LayerNorm(d_model)
        self.lm_head     = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, N = x.shape
        pos  = torch.arange(N, device=x.device).unsqueeze(0)
        h    = self.tok_emb(x) + self.pos_emb(pos)
        mask = nn.Transformer.generate_square_subsequent_mask(N, device=x.device)
        return self.lm_head(self.final_ln(self.transformer(h, mask=mask, is_causal=True)))

    @torch.no_grad()
    def generate(self, prompt, max_new_tokens=60):
        self.eval()
        curr = prompt.clone()
        for _ in range(max_new_tokens):
            if curr.size(1) >= self.max_seq_len: break
            curr = torch.cat([curr, self.forward(curr)[:,-1:].argmax(-1, keepdim=True)], 1)
        return curr


def count_params(m): return sum(p.numel() for p in m.parameters())
print('[OK] All model classes defined (memory-efficient gather active).')

In [ ]:
# ── Cell 4: Gather memory proof ──────────────────────────────────────
# Demonstrates that the fix reduces gather memory from O(N·N_total) to O(N·A)
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

    B, H, N_total, N, D = 4, 8, 1024, 1024, 64
    A_test = 32

    k_t  = torch.randn(B, H, N_total, D, device='cuda')
    v_t  = torch.randn(B, H, N_total, D, device='cuda')
    sel  = torch.randint(0, N_total, (B, N, A_test), device='cuda')

    # Run efficient gather
    kg, vg = _efficient_gather_kv(k_t, v_t, sel)
    torch.cuda.synchronize()

    peak_mb   = torch.cuda.max_memory_allocated() / 1e6
    naive_mb  = B * H * N * N_total * D * 4 / 1e6   # what the old code would allocate
    actual_mb = B * H * N * A_test * D * 4 / 1e6

    print(f'Naive O(N·N_total) gather would need : {naive_mb:.0f} MB per tensor')
    print(f'Efficient O(N·A) gather needs        : {actual_mb:.0f} MB per tensor')
    print(f'GPU peak memory actually used        : {peak_mb:.0f} MB')
    print(f'Memory reduction factor              : {naive_mb/actual_mb:.1f}x')
    print(f'kg shape: {kg.shape}  vg shape: {vg.shape}  [PASS]')
    del k_t, v_t, sel, kg, vg
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
else:
    print('CPU mode — memory proof skipped.')

In [ ]:
# ── Cell 5: Structured Dependency Dataset (seq_len = 1024) ───────────
import numpy as np

VOCAB   = 16
SEQ_LEN = 1024

def make_data(n=250, seq=SEQ_LEN):
    """
    Long-range dependency rules across 1024 tokens:
      Rule E->A : starts [1,1,1,1], pos 250-255=2, pos 750-755=2
      Rule X->Y : starts [6,6,6,6], pos 250-255=7, pos 750-755=7
      Rule B->Z : starts [3,3,3,3], pos 250-255=8, pos 750-755=8
    Filler tokens: {4,5,9,10,11}
    """
    data = np.random.choice([4,5,9,10,11], size=(n, seq)).astype(np.int64)
    rules = [(1,2),(6,7),(3,8)]
    for i in range(n):
        tok, dep = rules[i % 3]
        data[i,:4] = tok; data[i,250:255] = dep; data[i,750:755] = dep
    return torch.from_numpy(data[:,:-1]), torch.from_numpy(data[:,1:])

X_tr, Y_tr = make_data(200, SEQ_LEN)
X_te, Y_te = make_data(40,  SEQ_LEN)
print(f'Train: {X_tr.shape}   Test: {X_te.shape}   Vocab: {VOCAB}')

In [ ]:
# ── Cell 6: Build models and count parameters ─────────────────────────
D_MODEL, NHEAD, N_LAYERS, MAX_SEQ, GROUP = 128, 4, 4, SEQ_LEN, 2

dense  = DenseGPT(VOCAB, D_MODEL, NHEAD, N_LAYERS, MAX_SEQ)
asa32  = ASATransformerGPT(VOCAB, D_MODEL, NHEAD, N_LAYERS, MAX_SEQ, GROUP, max_a=32)
asa128 = ASATransformerGPT(VOCAB, D_MODEL, NHEAD, N_LAYERS, MAX_SEQ, GROUP, max_a=128)

print(f'Dense GPT          : {count_params(dense):,} params')
print(f'ASA-GPT (max_a=32) : {count_params(asa32):,} params')
print(f'ASA-GPT(max_a=128) : {count_params(asa128):,} params')

In [ ]:
# ── Cell 7: Training + benchmark utilities ───────────────────────────
import time

def _sync():
    if DEVICE.type == 'cuda': torch.cuda.synchronize()

def reset_vram():
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def peak_vram_mb():
    if DEVICE.type != 'cuda': return 0.0
    _sync()
    return torch.cuda.max_memory_allocated() / 1e6


def train_model(model, X, Y, *, epochs=3, bs=8, lr=3e-3, is_asa=False, max_a=None):
    model.train().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    N = len(X)
    for ep in range(epochs):
        perm = torch.randperm(N); loss_sum = 0; nb = 0
        t0 = time.perf_counter()
        for i in range(0, N, bs):
            bx = X[perm[i:i+bs]].to(DEVICE)
            by = Y[perm[i:i+bs]].to(DEVICE)
            opt.zero_grad()
            logits = model(bx, max_a=max_a) if is_asa else model(bx)
            F.cross_entropy(logits.view(-1, VOCAB), by.reshape(-1)).backward()
            opt.step()
            loss_sum += F.cross_entropy(logits.detach().view(-1, VOCAB), by.reshape(-1)).item()
            nb += 1
        print(f'  Epoch {ep+1}/{epochs} [{time.perf_counter()-t0:.2f}s]  loss={loss_sum/nb:.4f}')


def eval_perplexity(model, X, Y, *, is_asa=False, max_a=None, bs=8):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(X), bs):
            bx = X[i:i+bs].to(DEVICE); by = Y[i:i+bs].to(DEVICE)
            lg = model(bx, max_a=max_a) if is_asa else model(bx)
            total += F.cross_entropy(lg.view(-1, VOCAB), by.reshape(-1)).item()
            n += 1
    return math.exp(total / n)


def bench_generation(model, *, is_asa=False, max_a=None, gen_tok=60, runs=6):
    model.eval()
    prompt = torch.tensor([[1,1,1,1]], dtype=torch.int64, device=DEVICE)
    # warmup
    with torch.no_grad():
        if is_asa: model.generate(prompt, max_new_tokens=5, max_a=max_a)
        else:      model.generate(prompt, max_new_tokens=5)
    _sync()
    times = []
    for _ in range(runs):
        _sync(); t0 = time.perf_counter()
        with torch.no_grad():
            if is_asa: model.generate(prompt, max_new_tokens=gen_tok, max_a=max_a)
            else:      model.generate(prompt, max_new_tokens=gen_tok)
        _sync(); times.append(time.perf_counter() - t0)
    mu, std = float(np.mean(times)), float(np.std(times))
    return mu, std, gen_tok / mu


print('[OK] Utilities ready.')

In [ ]:
# ── Cell 8: Train and benchmark Standard Dense GPT ───────────────────
print('=' * 60)
print('  Standard Dense GPT')
print('=' * 60)
reset_vram()
train_model(dense, X_tr, Y_tr, epochs=3, bs=8, is_asa=False)
vram_train_dense = peak_vram_mb()
print(f'  VRAM peak (training) : {vram_train_dense:.1f} MB')

ppl_dense = eval_perplexity(dense, X_te, Y_te, is_asa=False)
reset_vram()
t_d, s_d, tps_d = bench_generation(dense, is_asa=False, gen_tok=60)
vram_gen_dense = peak_vram_mb()

print(f'  Perplexity      : {ppl_dense:.4f}')
print(f'  Gen time        : {t_d:.4f}s  ±{s_d:.5f}s')
print(f'  Tokens/sec      : {tps_d:.2f}')
print(f'  VRAM (gen)      : {vram_gen_dense:.1f} MB')
reset_vram(); torch.cuda.empty_cache()

In [ ]:
# ── Cell 9: Train and benchmark ASA-GPT (max_a=32) ──────────────────
print('=' * 60)
print('  ASA-GPT  max_a=32')
print('=' * 60)
reset_vram()
train_model(asa32, X_tr, Y_tr, epochs=3, bs=8, is_asa=True, max_a=32)
vram_train_32 = peak_vram_mb()
print(f'  VRAM peak (training) : {vram_train_32:.1f} MB')

ppl_32 = eval_perplexity(asa32, X_te, Y_te, is_asa=True, max_a=32)
reset_vram()
t_32, s_32, tps_32 = bench_generation(asa32, is_asa=True, max_a=32, gen_tok=60)
vram_gen_32 = peak_vram_mb()

print(f'  Perplexity      : {ppl_32:.4f}')
print(f'  Gen time        : {t_32:.4f}s  ±{s_32:.5f}s')
print(f'  Tokens/sec      : {tps_32:.2f}')
print(f'  VRAM (gen)      : {vram_gen_32:.1f} MB')
reset_vram(); torch.cuda.empty_cache()

In [ ]:
# ── Cell 10: Train and benchmark ASA-GPT (max_a=128) ─────────────────
print('=' * 60)
print('  ASA-GPT  max_a=128')
print('=' * 60)
reset_vram()
train_model(asa128, X_tr, Y_tr, epochs=3, bs=8, is_asa=True, max_a=128)
vram_train_128 = peak_vram_mb()
print(f'  VRAM peak (training) : {vram_train_128:.1f} MB')

ppl_128 = eval_perplexity(asa128, X_te, Y_te, is_asa=True, max_a=128)
reset_vram()
t_128, s_128, tps_128 = bench_generation(asa128, is_asa=True, max_a=128, gen_tok=60)
vram_gen_128 = peak_vram_mb()

print(f'  Perplexity      : {ppl_128:.4f}')
print(f'  Gen time        : {t_128:.4f}s  ±{s_128:.5f}s')
print(f'  Tokens/sec      : {tps_128:.2f}')
print(f'  VRAM (gen)      : {vram_gen_128:.1f} MB')
reset_vram(); torch.cuda.empty_cache()

In [ ]:
# ── Cell 11: Full comparative results table ───────────────────────────
rows = [
    ('Standard Dense GPT',  count_params(dense),  ppl_dense, vram_train_dense, t_d,   s_d,   tps_d,   vram_gen_dense),
    ('ASA-GPT (max_a=32)',  count_params(asa32),   ppl_32,   vram_train_32,   t_32,  s_32,  tps_32,  vram_gen_32),
    ('ASA-GPT (max_a=128)', count_params(asa128),  ppl_128,  vram_train_128,  t_128, s_128, tps_128, vram_gen_128),
]

W = 115
print('=' * W)
print(' BENCHMARK ESCALADO — 1024 TOKENS / ~500K PARAMS / GPU T4 — Dense GPT vs ASA-GPT ')
print('=' * W)
hdr = (f"{'MODELO':<23} | {'PARAMS':>10} | {'PERPLEXITY':>10} | "
       f"{'VRAM-TRAIN':>10} | {'GEN-TIME':>9} | {'STD':>8} | {'TOK/SEC':>9} | {'VRAM-GEN':>9}")
print(hdr)
print('-' * W)
for name, p, ppl, vtrain, t, std, tps, vgen in rows:
    print(f"{name:<23} | {p:>10,} | {ppl:>10.4f} | "
          f"{vtrain:>9.1f}M | {t:>9.4f}s | {std:>8.5f} | {tps:>9.2f} | {vgen:>8.1f}M")
print('=' * W)

# Relative metrics
ref_tps, ref_vtr, ref_vgen = rows[0][7], rows[0][3], rows[0][7]
print('\nSpeedup tok/sec vs Dense GPT:')
for name, *_, tps, vgen in rows:
    su = tps / rows[0][6]
    print(f'  {name:<23}: {su:.3f}x')

print('\nVRAM-Training savings vs Dense GPT:')
for name, _, __, vtrain, *_ in rows:
    sv = (1 - vtrain / rows[0][3]) * 100
    print(f'  {name:<23}: {vtrain:.1f} MB  ({sv:+.1f}%)')

print('\n[Done] Benchmark complete.')